# ASRA-Security — Kaggle Submission

AI Agent Security - Multi-Step Tool Attacks

This notebook writes `attack.py` to `/kaggle/working/` for evaluation.

In [ ]:
import inspect
import json
import os
import sys
from pathlib import Path

WORKING = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
INPUT = Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input"))
WORKING.mkdir(parents=True, exist_ok=True)

print("Working dir:", WORKING)
print("Input dir:", INPUT)
if INPUT.exists():
    print("Input contents:", [p.name for p in INPUT.iterdir()])

In [ ]:
# Install competition SDK from Kaggle dataset or pip fallback
sdk_paths = list(INPUT.glob("**/aicomp_sdk")) if INPUT.exists() else []
if sdk_paths:
    sys.path.insert(0, str(sdk_paths[0].parent))
    print("Using bundled SDK:", sdk_paths[0].parent)
else:
    try:
        import aicomp_sdk
        print("SDK already available:", aicomp_sdk.__file__)
    except ImportError:
        !pip install -q aicomp-sdk
        import aicomp_sdk
        print("Installed SDK:", aicomp_sdk.__file__)

In [ ]:
import aicomp_sdk
from aicomp_sdk.attacks.contracts import AttackAlgorithmBase, AttackCandidate, AttackRunConfig

sdk_info = {
    "version": getattr(aicomp_sdk, "__version__", "unknown"),
    "AttackAlgorithmBase": str(AttackAlgorithmBase),
    "AttackCandidate_fields": getattr(AttackCandidate, "__annotations__", {}),
    "AttackRunConfig_fields": getattr(AttackRunConfig, "__annotations__", {}),
}
print(json.dumps(sdk_info, indent=2))

In [ ]:
# Load attack.py from repo input dataset or local copy
source_candidates = [
    INPUT / "asra-security" / "attack.py",
    INPUT / "attack.py",
    Path("../attack.py"),
    Path("attack.py"),
]

attack_source = None
for candidate in source_candidates:
    if candidate.exists():
        attack_source = candidate.read_text(encoding="utf-8")
        print("Loaded attack source from:", candidate)
        break

if attack_source is None:
    raise FileNotFoundError("Could not locate attack.py in input paths")

out_path = WORKING / "attack.py"
out_path.write_text(attack_source, encoding="utf-8")
print(f"Wrote {out_path} ({len(attack_source)} bytes)")

In [ ]:
# Validate structure
spec = importlib.util.spec_from_file_location("attack_module", out_path)
import importlib.util
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

assert hasattr(module, "AttackAlgorithm"), "attack.py must define AttackAlgorithm"
assert issubclass(module.AttackAlgorithm, AttackAlgorithmBase)
print("Validation OK: AttackAlgorithm inherits from AttackAlgorithmBase")
print("run signature:", inspect.signature(module.AttackAlgorithm.run))

In [ ]:
# Optional local smoke test when sandbox env is available
try:
    from aicomp_sdk.agents import build_agent
    from aicomp_sdk.core.env import SandboxEnv
    from aicomp_sdk.guardrails.optimal import OptimalGuardrail

    env = SandboxEnv(seed=42, agent=build_agent("deterministic"), guardrail=OptimalGuardrail())
    algo = module.AttackAlgorithm()
    findings = algo.run(env, AttackRunConfig(time_budget_s=15, max_tool_hops=6))
    print(f"Smoke test complete: {len(findings)} candidate(s)")
    for i, finding in enumerate(findings[:5]):
        print(f"  [{i}] messages={len(finding.user_messages)}")
except Exception as exc:
    print("Smoke test skipped or failed:", exc)